## Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detection.


In [1]:
import os
from langchain_groq import ChatGroq

model = ChatGroq(
    model = "llama-3.3-70b-versatile"
)

## Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

* Long-running conversations that exceed context windows.
* Multi-turn dialogues with extensive history.
* Applications where preserving full conversation context matters.


In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

agents = create_agent(
    model = model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)


In [3]:
"""run with thread"""
config = {
    "configurable" : {
        "thread_id" : "test_1"
    }
}

In [4]:
questions = [
    "what is 4*12?",
    "What is 8*9?",
    "Where is Burj Khalifa?",
    "Founder of OLA?",
    "Which is the Most Widely used Web Framework in the world?",
    "In which year did the inagural duronto express run and between which two cities?"
]

In [5]:
for q in questions:
    response = agents.invoke(
        {
            "messages":[HumanMessage(content=q)]
        }, config=config
    )
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 4*12?', additional_kwargs={}, response_metadata={}, id='7c581829-090b-4112-9594-8faa488e4fb4'), AIMessage(content='4 * 12 = 48.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.015704465, 'completion_tokens_details': None, 'prompt_time': 0.007176783, 'prompt_tokens_details': None, 'queue_time': 0.162343696, 'total_time': 0.022881248}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd1f0-d887-7280-9744-f386d6b3eab0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='what is 4*12?', additional_kwargs={}, response_metadata={}, id='7c581829-090b-4112-9594-8faa488e4fb4'), AI

### Token Size MiddleWare

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $180/night, business center
3. Budget Stay - 3 star, $75/night, free wifi"""

agents = create_agent(
    model = model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger=("tokens", 200),
            keep=("tokens", 100)
        )
    ]
)

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

"""run with thread"""
config = {
    "configurable" : {
        "thread_id" : "test_1"
    }
}

In [ ]:

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agents.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

### **Human In the Loop MiddleWare**

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

* High-stakes operations requiring human approval (e.g. database writes, financial transactions).
* Compliance workflows where human oversight is mandatory.
* Long-running conversations where human feedback guides the agent.


llama-3.3-70b-versatile
